In [133]:
import os, sys
import json, yaml
import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import tbparse
from IPython.display import display

from dpvtex.dpvt_data import load_nicknames_dict

### Inject Parameters from Papermill

In [134]:
# Parameter Options
get_data_stats_from_name = False
use_relative_paths_to_output_dir = True

# Parameter Paths
working_dir = "."
output_dir = "_output/run.cpu_TODAY"
results_dir = "_output/run.cpu_TODAY/result_csvs_and_pdfs"
summary_csv_path = "_output/run.cpu_TODAY/result_csvs_and_pdfs/summary.csv"
config_path = "_output/run.cpu_TODAY/config.yaml"
data_nicknames_path = "_output/run.cpu_TODAY/data_nicknames.json"

In [136]:
if use_relative_paths_to_output_dir:
    results_dir = f"{output_dir}/result_csvs_and_pdfs"
    summary_csv_path = f"{output_dir}/result_csvs_and_pdfs/summary.csv"
    config_path = f"{output_dir}/config.yaml"
    data_nicknames_path = f"{output_dir}/data_nicknames.json"

In [ ]:
# Verify that paths exist
for path in [working_dir, output_dir, results_dir, summary_csv_path, config_path, data_nicknames_path]:
    if os.path.exists(path):
        print(f"file exists: {path}")
    else:
        print(f"file does NOT exist: {path}")

# Load and Format Data

In [138]:
def display_all(df, max_rows=None, max_cols=None):
    with pd.option_context('display.max_rows', max_rows, 'display.max_columns', max_cols):
        display(df)

In [139]:
def load_data(file_path, file_type=None):
    # if file type not given, infer from file extension
    if file_type == None:
        file_type = file_path.split('.')[-1]

    if file_type in ['json']:
        with open(file_path, 'r') as file:
            data = json.load(file)
        return data

    elif file_type in ['yaml', 'yml']:
        with open(file_path, "r") as file:
            data = yaml.safe_load(file)
        return data

    elif file_type in ['pickle', 'pkl', 'p']:
        with open(file_path, 'rb') as file:
            data = pickle.load(file)
        return data

    else:
        print(f'load failed: "{file_type}" file type not recognized.')

In [ ]:
# load config file
config_data = load_data(config_path)
config_yaml_str = yaml.dump(config_data, default_flow_style=False, sort_keys=False)
print(config_yaml_str)

data_pairs = dict(zip(config_data['train_data'], config_data['test_data']))

In [ ]:
# load data nicknames and absolute paths
data_nicknames_dict = load_nicknames_dict(data_nicknames_path)
for name,path in data_nicknames_dict.items():
    data_nicknames_dict[name] = f'{working_dir}/{path}'
for name,path in data_nicknames_dict.items():
    if os.path.exists(path):
        print(f"file exists: {name} => {path}")
    else:
        print(f"file does NOT exist: {name} => {path}")

In [ ]:
# load results summary
summary_df = pd.read_csv(summary_csv_path)
print(summary_df.columns)
display(summary_df)

In [143]:
# Function for pulling data stats from inspecting data.

def get_stats_from_data(data_path, take_first=True):
    '''
    Extracts the number of trees, leaves, and sites from a dataset.
    take_first: Takes the first tree as representative of the number of leaves and sites in the whole dataset.
                Otherwise, creates a min-max range over the all trees in the dataset.
    '''
    data_stats = {
        'num_trees': None,
        'num_leafs': [],
        'num_sites': [],
        'num_leafs_range': [np.inf, -np.inf],
        'num_sites_range': [np.inf, -np.inf],
    }
    data_dict = load_data(data_path)

    for i,(tree,vec) in enumerate(data_dict.items()):
        num_leafs = [len(tree.get_leaves())]
        num_sites = [len(x.sequence) for x in tree.get_leaves()]

        data_stats['num_leafs'] += num_leafs
        data_stats['num_sites'] += num_sites
        data_stats['num_leafs_range'][0] = min(data_stats['num_leafs_range'][0], np.min(num_leafs))
        data_stats['num_leafs_range'][1] = max(data_stats['num_leafs_range'][1], np.max(num_leafs))
        data_stats['num_sites_range'][0] = min(data_stats['num_sites_range'][0], np.min(num_sites))
        data_stats['num_sites_range'][1] = max(data_stats['num_sites_range'][1], np.max(num_sites))
        if take_first:
            break

    data_stats['num_trees'] = len(data_dict.items())
    data_stats['num_leafs'] = int(np.mean(data_stats['num_leafs']))
    data_stats['num_sites'] = int(np.mean(data_stats['num_sites']))
    return data_stats


def get_all_data_names_from_summary_df(summary_df):
    data_names = []
    data_names += list(set(summary_df.train_data))
    data_names += list(set(summary_df.test_data))
    data_names = list(set(data_names))
    return data_names


def build_data_stats_dict_from_summary_df(summary_df, take_first=True):
    '''
    Build dict of number of trees, leaves, and sites for each dataset in summary_df.
    '''
    data_stats = {}
    data_names = get_all_data_names_from_summary_df(summary_df)
    for i,data_name in enumerate(data_names):
        if i+1 % 5 == 0:
            print(f'i={i}')
        train_data_path = data_nicknames_dict[data_name]
        data_stats[data_name] = get_stats_from_data(train_data_path, take_first)

    return data_stats

In [145]:
# Functions for pulling data stats from name.

def extract_num_leaves(row):
    '''
    Extracts number of leaves from a row containing test or training data labels. This only works for alisim and perfect phylogeny data that follows specific naming rules and should eventually be replaces by something more sophisticated, e.g. adding num_leaves and other stats like that when creating initial csvs.
    '''
    if 'alisim' in row:
        return row.split('_')[1]  # Take the second entry after splitting by '_'
    elif 'leaf' in row:
        return row.split('leaf')[0]  # Take the part after 'leaf'
    elif 'perfect' in row:
        return row.split('_')[1]
    else:
        return None


def extract_num_sites(row):
    '''
    Extracts number of leaves from a row containing test or training data labels. This only works for alisim data that follows sepcific naming rules and should eventually be replaces by something more sophisticated, e.g. adding num_leaves and other stats like that when creating initial csvs.
    '''
    if 'alisim' in row:
        return row.split("_")[3]
    elif 'perfect' in row:
        return row.split('_')[7]
    else:
        return None


def extract_num_trees(row):
    '''
    Extracts number of leaves from a row containing test or training data labels. This only works for alisim data that follows sepcific naming rules and should eventually be replaces by something more sophisticated, e.g. adding num_leaves and other stats like that when creating initial csvs.
    '''
    if 'pp' in row:
        return row.split("trees")[0].split("_")[-1]
    elif 'perfect' in row:
        return row.split('_')[3]
    else:
        return None

In [ ]:
# Add summary columns
summary_df["label"] = summary_df[["model","train_data","test_data", "param_id"]].apply(lambda x:"\n".join(x), axis=1)
summary_df["percent_epochs"] = summary_df["train_epochs"]/summary_df["max_epochs"]
summary_df["model_and_train_data"] = summary_df["model"].astype(str) + "-" + summary_df["train_data"].astype(str)

if get_data_stats_from_name:
    summary_df["train_num_leaves"] = pd.to_numeric(summary_df["train_data"].apply(extract_num_leaves), errors="coerce")
    summary_df["train_num_sites"] = pd.to_numeric(summary_df["train_data"].apply(extract_num_sites), errors="coerce")
    summary_df["train_num_trees"] = pd.to_numeric(summary_df["train_data"].apply(extract_num_trees), errors="coerce")
    summary_df["test_num_leaves"] = pd.to_numeric(summary_df["test_data"].apply(extract_num_leaves), errors="coerce")
    summary_df["test_num_sites"] = pd.to_numeric(summary_df["test_data"].apply(extract_num_sites), errors="coerce")
    summary_df["test_num_trees"] = pd.to_numeric(summary_df["test_data"].apply(extract_num_trees), errors="coerce")
else:
    data_stats = build_data_stats_dict_from_summary_df(summary_df)
    for train_data,test_data in data_pairs.items():
        train_num_trees = data_stats[train_data]["num_trees"]
        test_num_trees = data_stats[test_data]["num_trees"]
    summary_df["train_num_leaves"] = [data_stats[x]["num_leafs"] for x in summary_df["train_data"]]
    summary_df["train_num_sites"] = [data_stats[x]["num_sites"] for x in summary_df["train_data"]]
    summary_df["train_num_trees"] = [data_stats[x]["num_trees"] for x in summary_df["train_data"]]
    summary_df["test_num_leaves"] = [data_stats[x]["num_leafs"] for x in summary_df["test_data"]]
    summary_df["test_num_sites"] = [data_stats[x]["num_sites"] for x in summary_df["test_data"]]
    summary_df["test_num_trees"] = [data_stats[x]["num_trees"] for x in summary_df["test_data"]]

# Sort models
model_order = ["TraverseMaxPooling", "TraverseAvgPooling", "TraverseNN"]
summary_df["model"] = pd.Categorical(summary_df["model"], categories=model_order, ordered=True)

num_runs = len(summary_df)
display_all(
    summary_df[["train_data", "train_num_leaves", "train_num_sites", "train_num_trees",
                "test_data", "test_num_leaves", "test_num_sites", "test_num_trees"]],
    max_rows=10,
)

In [147]:
# get dataframe from lightning log
def get_df_from_log(log_path):
    reader = tbparse.SummaryReader(log_path)
    return reader.scalars

In [148]:
# get dfs from logs
hyperparam_dfs = {}
train_dfs = {}
test_dfs = {}
custom_dfs = {}

for index,row in summary_df.iterrows():
    label_str = row.label
    label = (row.model,row.train_data,row.test_data,row.param_id)
    train_df = get_df_from_log(row.train_llog_path)
    train_dfs[label] = train_df
    test_df = get_df_from_log(row.test_llog_path)
    test_dfs[label] = test_df
    custom_df = get_df_from_log(row.train_clog_path)
    custom_dfs[label] = custom_df
    hyperparam_dfs[label] = {}
    for i in range(row.n_hyperparam_trials):
        hyperparam_dfs[label][i] = get_df_from_log(f'{row.hyper_llog_path}/version_{i}')


In [ ]:
# reshape data
models = ['model', 'train_data', 'test_data', 'param_id']
hyperparams = ['learning_rate', 'batch_size', 'accum_grad_batches', 'max_epochs', 'feature_length', 'dim_mlp_layers']

# only keep one row per train_data

# train_df = summary_df
# train_df['label'] = train_df['model'].astype(str) + "-" + train_df['train_data'].astype(str)
train_df = summary_df.groupby('model_and_train_data')[hyperparams].first().reset_index()

melt_df = pd.melt(train_df, id_vars='model_and_train_data', var_name='Metric', value_name='Value')
melt_df = melt_df[melt_df.Metric.isin(hyperparams)]
display(melt_df)

In [150]:
# Ensures plt.subplots() is iterable in the 1x1 base case.
def plt_subplots(*args, **kwargs):
    fig, axs = plt.subplots(*args, **kwargs)
    if not isinstance(axs, np.ndarray):  # Check if it's a single Axes instance
        axs = np.array([axs])
    return fig, axs

# Functions for label formatting (converts to sci formatting for numbers outside given range, truncates to set significant digits)
def truncate_to_significant_digits(x, sig_digits):
    if x == 0:
        return 0
    else:
        magnitude = np.floor(np.log10(abs(x)))
        factor = 10**(sig_digits - magnitude - 1)
        truncated_value = np.floor(x * factor) / factor
        return truncated_value

def num_formatter(x, sig_digits=4, sci_range=[1e-6, 1e6]):
    x = truncate_to_significant_digits(x, sig_digits)
    if (x != 0) and (x < sci_range[0] or x > sci_range[1]):
        return "{:.{}e}".format(x, sig_digits)
    return x

# Testing performance (AUROC and test loss)

In [ ]:
def build_heatmap(df, value_name, title=None, v_range=[None, None]):
    # format data
    df_sorted = df.sort_values(by=["model", "train_num_leaves", "train_num_sites", "train_num_trees"])
    indices = ["model"]
    extra_cols = ["train_num_leaves", "train_num_sites", "train_num_trees"]
    for col_name in extra_cols:
        if not df_sorted[col_name].isnull().values.any():
            indices.append(col_name)
    if len(indices) > 1:
        heatmap_data = df_sorted.pivot_table(
            index=indices,
            columns=["test_num_leaves", "test_num_sites"],
            values=value_name
        )
    else:
        heatmap_data = df_sorted.pivot(
            index="model",
            columns="test_data",
            values=value_name
        )

    # build heatmap
    fig_width = (len(heatmap_data.columns) * 0.75) + 4
    fig_height = (len(heatmap_data) * 0.25) + 4
    plt.figure(figsize=(fig_width, fig_height))
    ax = sns.heatmap(
        data=heatmap_data,
        annot=True,
        cbar_kws={"label": f"{value_name}"},
        vmin=v_range[0], vmax=v_range[1],
    )

    # axis labels
    plt.xticks(rotation=90, ha="right")
    ax.set_xlabel("Test Dataset\nnumber of leaves - number of sites")
    ax.set_ylabel("Training Dataset\nmodel - number of leaves - number of sites - number of trees")
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(f"{results_dir}/{value_name}_heatmap.pdf")
    plt.show()


# heatmap of AUROCs
heatmap_settings = {
  "test_auroc": {
      "title": "Test AUROC",
      "v_range": [0.0, 1.0],
      "train_fields": ["model", "train_num_leaves", "train_num_sites", "train_num_trees"],
      "test_fields": ["test_num_leaves", "test_num_sites"],
  },
  "test_loss": {
      "title": "Test Loss",
      "v_range": [0.0, None],
      "train_fields": ["model", "train_num_leaves", "train_num_sites", "train_num_trees"],
      "test_fields": ["test_num_leaves", "test_num_sites"],
  },
}

for value_name,settings in heatmap_settings.items():
    build_heatmap(
        df=summary_df,
        value_name=value_name,
        title=settings['title'],
        v_range=settings['v_range'],
    )


In [ ]:
# barplot hyperparameters combined
df = melt_df
print(df.columns)
if len(df['model_and_train_data'].unique()) < 10:
    num_params = len(hyperparams)
    plt.figure(figsize=(3 * num_params,8))
    sns.barplot(x='Metric', y='Value', hue='model_and_train_data', data=df, palette='deep')
    # plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.4), ncol=num_runs/2)
    plt.xticks(rotation=45)
    plt.title('Hyper Parameters by Model')
    plt.tight_layout()
    plt.savefig(f'{results_dir}/hyperparameter_summary.barplot.pdf')
    plt.show()
    plt.clf()
else:
    print("To many trained models, don't plot hyperparameter summary")

## Training and Hyperparameter settings and results

In [ ]:
# individual bar plots
x_measures = {
  "Training Runtime (in secs)": "train_walltime",
  "Learning Rate": "learning_rate",
  "Batch Size": "batch_size",
  "Gradient Accumulation": "accum_grad_batches",
  "Max Epochs": "max_epochs",
  "Train Epochs": "train_epochs",
  "Train Steps": "train_steps",
  "Percent of Max Epochs Used": "percent_epochs",
}

model_list = summary_df['model'].unique()

for x_title,x_col in x_measures.items():
    fig, axes = plt_subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
    x_pad_factor = 0.05
    x_pad = abs(summary_df[x_measures[x_title]].min() - summary_df[x_measures[x_title]].max()) * x_pad_factor
    print(x_pad)
    x_min = summary_df[x_measures[x_title]].min() - x_pad
    x_max = summary_df[x_measures[x_title]].max() + x_pad
    for ax, model in zip(axes, model_list):
        this_df = summary_df[summary_df['model']==model]
        sns.barplot(y="train_data", x=f"{x_col}", data=this_df, palette='deep', hue='train_data', ax=ax)
        ax.set_ylabel(f"Train Data")
        ax.set_xlabel(f"{x_title}")
        ax.set_title(f"{model}")
        ax.set_xlim(x_min, x_max)
        # ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=False, prune='both'))
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(num_formatter))
        labels = ax.get_xticklabels()
        for i in range(len(labels)):
            labels[i].set_rotation(90)
        # ax.set_xticklabels(labels)

    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f'{results_dir}/{x_col}.barplot.pdf')
    plt.show()
    plt.clf()


# More runtime plots

In [ ]:
# line plots over epochs (using custom logs)
measures = {
  "Runtime over Batches": "walltime_per_batch",
  "Loss over Batches": "loss_per_batch",
  "Runtime over Epochs": "walltime_per_epoch",
  "Loss over Epochs": "avgloss_per_epoch",
}

df_list = []
for key,df in custom_dfs.items():
  this_df = df
  df['model_and_train_data'] = key[0] + "-" + key[1]
  df['model'] = key[0]
  df_list.append(this_df)

runtime_df = pd.concat(df_list)

for title,col in measures.items():
  fig, axes = plt_subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
  this_tag_df = runtime_df[runtime_df.tag==col]
  x_min = 0
  x_max = this_tag_df["step"].max() + 2
  for ax, model in zip(axes, model_list):
      this_df = this_tag_df[this_tag_df['model']==model]
      plot = sns.lineplot(y="value", x="step", data=this_df, hue="model_and_train_data", palette='deep', ax=ax, legend=True)
      ax.set_xlabel("Training Step")
      ax.set_ylabel(f"{title}")
      ax.set_title(f"{model}")
      ax.set_xlim(x_min, x_max)
      plt.tight_layout()
      handles, labels = plot.get_legend_handles_labels()  # Get legend info from subplot
      ax.get_legend().remove()
  fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5))

  plt.savefig(f'{results_dir}/{col}.barplot.pdf')
  plt.show()
  plt.clf()

## Loss over walltime

In [ ]:
# line plot loss over walltime (using custom logs)
fig, axes = plt_subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
model_list = list(model_list)
label_set = set()
for key,df in custom_dfs.items():
    label = key[0] + "-" + key[1]
    if label in label_set:
        continue
    label_set.add(label)
    df_time = df[df.tag == 'walltime_per_batch']
    df_loss = df[df.tag == 'loss_per_batch']
    ax = axes[model_list.index(key[0])]
    ax.plot(df_time.value, df_loss.value, label=key[1])
    ax.set_title(key[0])

fig.supylabel("Training loss", fontsize=12)
fig.supxlabel('Walltime (in seconds)', fontsize=12)

x_title = "Training Loss over Time"
plt.xticks(rotation=45)

fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig(f'{results_dir}/loss_per_walltime.lineplot.pdf')
plt.show()
plt.clf()


## Training loss, positive, and negative predictions over epochs

In [ ]:
# line plots over epochs (using lightning logs)
x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

for x_title,x_col in x_measures.items():
    fig, axes = plt_subplots(nrows=1, ncols=len(model_list), figsize=(15, 6), sharey=True)
    label_set = set()
    for key,df in train_dfs.items():
        label = key[0] + "-" + key[1]
        if label in label_set:
            continue
        label_set.add(label)
        ax = axes[model_list.index(key[0])]
        this_df = df[df.tag == x_col]
        ax.plot(this_df.step, this_df.value, label=key[1])
        ax.set_title(key[0])

    fig.supylabel(f"{x_title}", fontsize=12)
    fig.supxlabel('Epochs', fontsize=12)
    fig.suptitle(f"{x_title} by Model")

    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.savefig(f'{results_dir}/{x_col}.lineplot.pdf')
    plt.show()
    plt.clf()

## Training loss over epochs in hyperparameter trials

In [ ]:
# line plot hyperparam training
x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

label_set = set()
for x_title,x_col in x_measures.items():
    for label,hp_dict in hyperparam_dfs.items():
      model_and_train_label = label[0] + "-" + label[1]
      if model_and_train_label in label_set:
        continue
      label_set.add(model_and_train_label)
      label_str = ','.join(label)
      for key,df in hp_dict.items():
          df_tag = df[df.tag == x_col]
          plt.plot(df_tag.step, df_tag.value, label=key)

      plt.xlabel("Epochs")
      plt.ylabel(f"{x_title}")
      plt.title(f"{x_title} by Model\n{model_and_train_label}")
      plt.xticks(rotation=45)

      plt.legend(loc="upper right")
      plt.tight_layout()
      plt.savefig(f'{results_dir}/hyperparameter_opt.{x_col}.lineplot.pdf')
      plt.show()
      plt.clf()